### Logic flow
* per patient: load pt_neurs_info_df, pt_neur_trigs_df, pt_beh_trials_df
    * pair trials with triggers: psychopy row i = i-th trigger trial
    * per epoch: extract spikes & FRs per trial & neuron, then stack patients along the neuron axis -> pseudopopulation
* save per epoch: spikes (trials, all_neurs), FRs (trials, all_neurs, bins; raw — norm downstream via flag), trial_mean_FRs (trials, all_neurs), bins (bins,)
* also save all_neurs_info_df (neuron metadata: patient, region, stacked column idx) and per-patient trial tables

In [1]:
import pandas as pd, numpy as np, glob, os
from scipy.ndimage import gaussian_filter1d
from neo.io import BlackrockIO

In [2]:
pseudopop_dir = '../../outputs/processed_data/pseudopop'; os.makedirs(pseudopop_dir, exist_ok=True)
processed_dir = '../../outputs/processed_data'  # per-patient inputs (neurs_info_df.parquet)

all_beh_trials_df = pd.read_csv('../../data/psychopy/all_beh_trials_df.csv')
patients = all_beh_trials_df.loc[all_beh_trials_df['eligibility'] == 'neuronal', 'subj'].unique(); print(f'neural patients: {patients}')

size, dt = .02, .01 # smoothing params

# set epoch prestarts and durations (xlims)
epochs = ['baseline', 'stim', 'delay', 'response', 'feedback']; epoch_prestarts, epoch_durs = {}, {}

for epoch in epochs:

    # where to start x-axis
    if epoch == 'baseline': epoch_prestarts[epoch] = -0
    elif epoch == 'response': epoch_prestarts[epoch] = -1 # 1s before response submitted
    else: epoch_prestarts[epoch] = -.25

    # plot duration
    if epoch == 'delay': epoch_durs[epoch] = 1.5
    elif epoch == 'response': epoch_durs[epoch] = 0 # 0s after response submitted
    else: epoch_durs[epoch] = 1

print(f'prestarts: {epoch_prestarts}\ndurations: {epoch_durs}')

neural patients: [12. 18. 21. 22.]
prestarts: {'baseline': 0, 'stim': -0.25, 'delay': -0.25, 'response': -1, 'feedback': -0.25}
durations: {'baseline': 1, 'stim': 1, 'delay': 1.5, 'response': 0, 'feedback': 1}


### helpers

In [3]:
def get_pt_trigs(patient):
    ''' digital trigger stream from the nev file, times relative to block1 start '''
    nev_file = glob.glob(f'../../data/20{int(patient)}/raw/*.nev')[0]; io = BlackrockIO(nev_file)
    seg = io.read_block(lazy=False).segments[0]; dig_ev = [ev for ev in seg.events if "digital" in ev.name.lower()][0]

    code_map = {10:"block started",20:"baseline started",30:"stim started",40:"delay started",50:"task started",51:"marker moved",52:"left pressed",53:"left released",54:"right pressed",55:"right released",56:"response submitted",60:"anticipation started",70:"feedback started",80:"block ended"}

    pt_neur_trigs_df_raw = pd.DataFrame({"trigger_code": dig_ev.labels.astype(int),"time": dig_ev.times.magnitude}); pt_neur_trigs_df_raw["event"] = pt_neur_trigs_df_raw["trigger_code"].map(code_map)

    # align times relative to block1 start; drop triggers before it and abs time
    block1_start_idx = pt_neur_trigs_df_raw.index[pt_neur_trigs_df_raw["trigger_code"] == 10][0]; pt_neur_trigs_df_raw["rel_time"] = pt_neur_trigs_df_raw["time"] - pt_neur_trigs_df_raw.loc[block1_start_idx, "time"]
    pt_neur_trigs_df = pt_neur_trigs_df_raw[block1_start_idx:]; pt_neur_trigs_df = pt_neur_trigs_df.drop(columns=['time']).reset_index(drop=True)
    return pt_neur_trigs_df


def get_epoch_spikes_and_FRs(pt_neur_trigs_df, neurs_info_df, epoch, size=size, dt=dt):
    ''' for each trial and neuron, get spike times and smoothed FRs (Hz) in epoch window
        pt_neur_trigs_df: trig times, neurs_info_df: spike times per neur '''

    epoch_prestart, epoch_dur = epoch_prestarts[epoch], epoch_durs[epoch]
    bin_edges = np.arange(epoch_prestart, epoch_dur + dt, dt); bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2; n_bins = len(bin_edges) - 1

    # get epoch start indices & times
    if epoch != 'response': epoch_start_rows = pt_neur_trigs_df[pt_neur_trigs_df['event'] == f'{epoch} started'].index
    # using anticipation started instead of response submitted bc sometimes they dont respond
    else: epoch_start_rows = pt_neur_trigs_df[pt_neur_trigs_df['event'] == 'anticipation started'].index
    epoch_starts = pt_neur_trigs_df.loc[epoch_start_rows, 'rel_time'].values; n_trials = len(epoch_starts)

    trial_neur_spikes, trial_neur_FRs = np.empty((n_trials, len(neurs_info_df)), dtype=object), np.zeros((n_trials, len(neurs_info_df), n_bins))

    for trial_idx in range(n_trials):
        for neur_idx, (_, neur_row) in enumerate(neurs_info_df.iterrows()):

            trial_epoch_spikes = neur_row['spikes'][(neur_row['spikes'] >= epoch_starts[trial_idx] + epoch_prestart) &
                                                    (neur_row['spikes'] <= epoch_starts[trial_idx] + epoch_dur)]
            trial_epoch_spikes = trial_epoch_spikes - epoch_starts[trial_idx] # align

            # bin and smooth
            counts, _ = np.histogram(trial_epoch_spikes, bins=bin_edges)
            smooth_spike_train = gaussian_filter1d(counts.astype(float), sigma=size/dt, mode='reflect', truncate=3.0)
            smooth_spike_train = smooth_spike_train / dt  # instantaneous FR (Hz): spikes/bin / s/bin

            trial_neur_spikes[trial_idx, neur_idx], trial_neur_FRs[trial_idx, neur_idx, :] = trial_epoch_spikes, smooth_spike_train

    return trial_neur_spikes, trial_neur_FRs, bin_centers


### per-patient extraction

In [4]:
pt_beh_trials_dfs, pt_neurs_info_dfs, pt_epoch_data = {}, [], {}
for patient in patients:

    pt_neurs_info_df = pd.read_parquet(f'{processed_dir}/20{int(patient)}/neurs_info_df.parquet')
    pt_neur_trigs_df = get_pt_trigs(patient)

    # triggers arrive in played order, so pair against trials in played order: trigger trial i <-> i-th trial played
    pt_beh_trials_df = all_beh_trials_df.loc[all_beh_trials_df['subj'] == patient].sort_values('trial_chronological_idx').reset_index(drop=True)
    pt_epoch_data[patient] = {epoch: get_epoch_spikes_and_FRs(pt_neur_trigs_df, pt_neurs_info_df, epoch) for epoch in epochs}

    # played order differs across patients (blocks shuffled), so put trials in trial_key order -> one shared trial axis for stacking
    trial_key_order = np.argsort(pt_beh_trials_df['trial_key'].values)
    pt_epoch_data[patient] = {epoch: (spikes[trial_key_order], FRs[trial_key_order], bins) for epoch, (spikes, FRs, bins) in pt_epoch_data[patient].items()}
    pt_beh_trials_dfs[patient] = pt_beh_trials_df.iloc[trial_key_order].reset_index(drop=True)

    pt_neurs_info_dfs.append(pt_neurs_info_df)
    pt_neur_trigs_df.to_parquet(f'{processed_dir}/20{int(patient)}/neur_trigs_df.parquet')
    print(f'pt{int(patient)}: {len(pt_neurs_info_df)} neurons, {len(pt_beh_trials_df)} trials, {len(pt_neur_trigs_df)} triggers')

# every patient must now carry the identical trial sequence, or the neuron-axis stack is meaningless
for patient in patients[1:]:
    assert (pt_beh_trials_dfs[patient]['trial_key'].values == pt_beh_trials_dfs[patients[0]]['trial_key'].values).all()
    assert (pt_beh_trials_dfs[patient]['true_stim'].round(2).values == pt_beh_trials_dfs[patients[0]]['true_stim'].round(2).values).all()

pt12: 23 neurons, 240 trials, 2454 triggers


pt18: 13 neurons, 240 trials, 2546 triggers


pt21: 14 neurons, 240 trials, 2449 triggers


pt22: 7 neurons, 240 trials, 2344 triggers


### stack patients along the neuron axis -> pseudopopulation

In [5]:
# neuron metadata: which stacked column is which neuron
all_neurs_info_df = pd.concat(pt_neurs_info_dfs, ignore_index=True)
all_neurs_info_df['pop_neur_idx'] = np.concatenate([np.arange(len(df)) for df in pt_neurs_info_dfs])  # column within own patient
all_neurs_info_df['col_idx'] = np.arange(len(all_neurs_info_df))                                      # column in stacked matrices
all_neurs_info_df.to_parquet(f'{pseudopop_dir}/all_neurs_info_df.parquet')
print(f'all_neurs_info_df: {len(all_neurs_info_df)} neurons across {all_neurs_info_df["patient"].nunique()} patients')
all_neurs_info_df[['patient', 'chanID', 'unitID', 'region', 'pop_neur_idx', 'col_idx']].head()

all_neur_df: 57 neurons across 4 patients


,patient,chanID,unitID,region,pop_neur_idx,col_idx
0,12,97,612,mLOFC1,0,0
1,12,98,1583,mLOFC2,1,1
2,12,99,703,mLOFC3,2,2
3,12,101,952,mLOFC5,3,3
4,12,102,2460,mLOFC6,4,4


In [6]:
for epoch in epochs:
    os.makedirs(f'{pseudopop_dir}/{epoch}', exist_ok=True)
    spikes = np.hstack([pt_epoch_data[pt][epoch][0] for pt in patients])                  # (trials, all_neurs) spike times
    FRs    = np.concatenate([pt_epoch_data[pt][epoch][1] for pt in patients], axis=1)     # (trials, all_neurs, bins)
    bins   = pt_epoch_data[patients[0]][epoch][2]
    trial_mean_FRs = FRs.mean(axis=2)  # scalar FR per (trial, neuron); feeds tuning heatmaps & decoding

    np.save(f'{pseudopop_dir}/{epoch}/spikes.npy', spikes, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/FRs.npy', FRs, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/trial_mean_FRs.npy', trial_mean_FRs, allow_pickle=True)
    np.save(f'{pseudopop_dir}/{epoch}/bin_centers.npy', bins, allow_pickle=True)
    print(f'{epoch}: spikes {spikes.shape}, FRs {FRs.shape}, trial_mean_FRs {trial_mean_FRs.shape}, bins {bins.shape}')

# no trial table saved: rows are trial_key order, so downstream just sorts all_beh_trials_df.csv by trial_key

baseline: spikes (240, 57), FRs (240, 57, 100), trial_mean_FRs (240, 57), bins (100,)
stim: spikes (240, 57), FRs (240, 57, 125), trial_mean_FRs (240, 57), bins (125,)
delay: spikes (240, 57), FRs (240, 57, 175), trial_mean_FRs (240, 57), bins (175,)
response: spikes (240, 57), FRs (240, 57, 100), trial_mean_FRs (240, 57), bins (100,)


feedback: spikes (240, 57), FRs (240, 57, 125), trial_mean_FRs (240, 57), bins (125,)
